In [248]:
import gym
import numpy as np
import math
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from envs import TFAugmentedFrozenLakeEnv

In [566]:
env = TFAugmentedFrozenLakeEnv(
    n_thought_states=1,
    n_thought_acts=4,
    d_model=8,
)

In [567]:
env.pos_embedding.weight.data = (
    env.pos_embedding.weight.data
    / torch.norm(env.pos_embedding.weight.data, p=2, dim=-1, keepdim=True)
)
env.action_embedding.weight.data = (
    env.action_embedding.weight.data
    / torch.norm(env.action_embedding.weight.data, p=2, dim=-1, keepdim=True)
)

In [568]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=10):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # shape: (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (batch_size, seq_len, d_model)
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]

def init_weights(module):
    # Linear layers
    if isinstance(module, nn.Linear):
        nn.init.xavier_uniform_(module.weight)
        if module.bias is not None:
            nn.init.zeros_(module.bias)

    # Multihead attention (Q, K, V projections + output proj)
    elif isinstance(module, nn.MultiheadAttention):
        nn.init.xavier_uniform_(module.in_proj_weight)
        if module.in_proj_bias is not None:
            nn.init.zeros_(module.in_proj_bias)

        nn.init.xavier_uniform_(module.out_proj.weight)
        if module.out_proj.bias is not None:
            nn.init.zeros_(module.out_proj.bias)


In [569]:
env_states = torch.arange(env.n_states)[:, None]
init_thought_state = env.initial_thought.detach()
init_thought_state = init_thought_state / torch.norm(init_thought_state, p=2)

In [570]:
init_joint_state = torch.cat((env_states, torch.tile(init_thought_state[None], (env.n_states, 1))), dim=1)

In [571]:
thought_actions = torch.arange(env.n_acts + 1, env.n_acts + 1 + env.n_thought_acts)

In [572]:
env.action_embedding(thought_actions).shape

torch.Size([4, 8])

In [573]:
curr_joint_state = init_joint_state.detach()
curr_joint_state = torch.tile(curr_joint_state, (env.n_thought_acts, 1))
curr_thought_actions = torch.tile(thought_actions[:, None], (1, 16)).reshape(-1)

In [574]:
curr_joint_state.shape, curr_thought_actions.shape

(torch.Size([64, 9]), torch.Size([64]))

In [575]:
pos_enc = PositionalEncoding(env.d_model, 100)

In [576]:
env_embed = env.pos_embedding(curr_joint_state[:, 0].int())
act_embed = env.action_embedding(curr_thought_actions)

In [577]:
encoder_layer = nn.TransformerEncoderLayer(
    d_model=env.d_model,
    nhead=2,
    dropout=0.0,
    dim_feedforward=env.d_model * 4,
    # activation="gelu",
    batch_first=True,
    norm_first=True,
)
transformer = nn.TransformerEncoder(encoder_layer, num_layers=1, enable_nested_tensor=False)
transformer.apply(init_weights)

TransformerEncoder(
  (layers): ModuleList(
    (0): TransformerEncoderLayer(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=8, out_features=8, bias=True)
      )
      (linear1): Linear(in_features=8, out_features=32, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
      (linear2): Linear(in_features=32, out_features=8, bias=True)
      (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout1): Dropout(p=0.0, inplace=False)
      (dropout2): Dropout(p=0.0, inplace=False)
    )
  )
)

In [578]:
seq = torch.cat((
    env_embed[:, None],
    act_embed[:, None],
    curr_joint_state[:, 1:][:, None],
), dim=1)
next_thought_state = transformer(
    seq
    # pos_enc(seq)
)

In [579]:
next_thought_state.shape

torch.Size([64, 3, 8])

In [580]:
next_thought_state = next_thought_state[:, 2]
# next_thought_state = env.lnorm(next_thought_state)
next_thought_state = next_thought_state / torch.norm(next_thought_state, p=2, dim=1, keepdim=True)

In [581]:
thought_dot_prod = next_thought_state @ next_thought_state.T
thought_dot_prod.max(), thought_dot_prod.min()

(tensor(1.0000, grad_fn=<MaxBackward1>),
 tensor(-0.5119, grad_fn=<MinBackward1>))

In [582]:
off_diagonal_mask = torch.triu(torch.ones_like(thought_dot_prod), 1)
torch.triu(thought_dot_prod, 1).sum() / off_diagonal_mask.sum()

tensor(0.5814, grad_fn=<DivBackward0>)

In [339]:
env_dot_prod = env_embed[:16] @ env_embed[:16].T
env_dot_prod.max(), env_dot_prod.min()

(tensor(1.0000), tensor(-0.1969))

In [340]:
off_diagonal_mask = torch.triu(torch.ones_like(env_dot_prod), 1)
torch.triu(env_dot_prod, 1).sum() / off_diagonal_mask.sum()

tensor(-0.0102)

In [193]:
act_dot_prod = act_embed[::16] @ act_embed[::16].T
act_dot_prod.max(), act_dot_prod.min()

(tensor(1.0000), tensor(-0.0652))

In [194]:
off_diagonal_mask = torch.triu(torch.ones_like(act_dot_prod), 1)
torch.triu(act_dot_prod, 1).sum() / off_diagonal_mask.sum()

tensor(0.0287)

In [ ]:
assert 0

## Old

In [2]:
n_thought_acts = 3
n_thought_states = 10
d_model = 16
seed = 42

In [3]:
env = TFAugmentedGridWorldEnv2(
    n_thought_states=n_thought_states,
    n_thought_acts=n_thought_acts,
    n_goals=2,
    deterministic_start=True,
    d_model=d_model,
    seed=seed,
)

In [4]:
obs = env.reset()

In [5]:
obs

{'letter': np.int64(1),
 'position': [3, 3],
 'thought': tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])}

In [6]:
next_obss = []
for act in range(5 + n_thought_acts):
    _ = env.reset()
    next_obs, rew, done, _ = env.step(act)
    next_obss.append(
        np.concatenate([np.array([v], dtype=np.float32) if k == "letter" else np.array(v, dtype=np.float32) for k, v in next_obs.items()], axis=0)
    )

/tmp/ipykernel_1065480/2551814597.py:6: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  np.concatenate([np.array([v], dtype=np.float32) if k == "letter" else np.array(v, dtype=np.float32) for k, v in next_obs.items()], axis=0)


In [7]:
next_obss

[array([2., 3., 3., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.], dtype=float32),
 array([1., 2., 3., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.], dtype=float32),
 array([1., 4., 3., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.], dtype=float32),
 array([1., 3., 2., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.], dtype=float32),
 array([2., 3., 4., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0.], dtype=float32),
 array([ 1.        ,  3.        ,  3.        ,  0.46524566, -0.20657514,
        -0.3988726 , -0.18239205,  0.09476374,  0.44416705, -0.09531733,
         0.04156468, -0.27183622, -0.18161179,  0.4390949 , -0.4735986 ,
         0.3205355 ,  0.2652688 ,  0.4504122 , -0.24278216], dtype=float32),
 array([ 1.        ,  3.        ,  3.        , -0.27866328,  0.02849502,
        -0.40749925, -0.32763   , -0.23626818,  0.2109445 ,  0.1601114 ,
        -0.36

In [8]:
np.linalg.matrix_rank(next_obss)

np.int64(6)

In [9]:
_ = env.reset()
for _ in range(10):
    act = np.random.randint(5, 8)
    next_obs, rew, done, _ = env.step(act)
    print(act, next_obs)

5 {'letter': np.int64(2), 'position': [3, 3], 'thought': tensor([-0.0756,  0.4006, -0.2252,  0.3733, -0.3599, -0.1275, -0.1651, -0.1831,
        -0.1811, -0.2048,  0.1270, -0.2558,  0.3327,  0.2509, -0.0109, -0.3041])}
5 {'letter': np.int64(2), 'position': [3, 3], 'thought': tensor([-0.0756,  0.4006, -0.2252,  0.3733, -0.3599, -0.1275, -0.1651, -0.1831,
        -0.1811, -0.2048,  0.1270, -0.2558,  0.3327,  0.2509, -0.0109, -0.3041])}
7 {'letter': np.int64(2), 'position': [3, 3], 'thought': tensor([-0.0756,  0.4006, -0.2252,  0.3733, -0.3599, -0.1275, -0.1651, -0.1831,
        -0.1811, -0.2048,  0.1270, -0.2558,  0.3327,  0.2509, -0.0109, -0.3041])}
5 {'letter': np.int64(2), 'position': [3, 3], 'thought': tensor([-0.0756,  0.4006, -0.2252,  0.3733, -0.3599, -0.1275, -0.1651, -0.1831,
        -0.1811, -0.2048,  0.1270, -0.2558,  0.3327,  0.2509, -0.0109, -0.3041])}
7 {'letter': np.int64(2), 'position': [3, 3], 'thought': tensor([-0.0756,  0.4006, -0.2252,  0.3733, -0.3599, -0.1275, -0.16

In [10]:
env.thought_state_embedding.weight @ env.thought_state_embedding.weight.T

tensor([[ 0.9853,  0.1676,  0.1450,  0.1315, -0.1479,  0.1126, -0.1901,  0.5625,
          0.1934,  0.3266],
        [ 0.1676,  1.1225, -0.4512, -0.3640, -0.3535, -0.2992,  0.2221,  0.2527,
         -0.2748,  0.4015],
        [ 0.1450, -0.4512,  1.4548, -0.3283, -0.2635,  0.6973,  0.1870,  0.0210,
          0.3630,  0.5025],
        [ 0.1315, -0.3640, -0.3283,  1.1275, -0.0815, -0.4468, -0.0888, -0.2139,
         -0.0447, -0.2639],
        [-0.1479, -0.3535, -0.2635, -0.0815,  0.8634, -0.0472, -0.1117,  0.0556,
          0.0932, -0.5424],
        [ 0.1126, -0.2992,  0.6973, -0.4468, -0.0472,  1.5890,  0.8646, -0.0985,
          0.7088,  0.0106],
        [-0.1901,  0.2221,  0.1870, -0.0888, -0.1117,  0.8646,  1.8710, -0.2938,
          0.7573,  0.0505],
        [ 0.5625,  0.2527,  0.0210, -0.2139,  0.0556, -0.0985, -0.2938,  1.3132,
          0.1067,  0.0509],
        [ 0.1934, -0.2748,  0.3630, -0.0447,  0.0932,  0.7088,  0.7573,  0.1067,
          1.0633,  0.3197],
        [ 0.3266,  

In [11]:
env.thought_state_embedding.weight.shape

torch.Size([10, 16])